# 01b · MEPS forecasts at the station (2017–2021)
Extracts the MEPS numerical weather prediction at the grid point nearest to the station from the MET Norway THREDDS archive and writes `data/nwp/meps_helsinki_00Z.csv`. The table is already included in the repository; this notebook only needs to be run to rebuild it (about 2–3 hours).

| | before 2020-02-04 | from 2020-02-04 |
|---|---|---|
| Irradiance | `meps_mbr0_full_backup` | `meps_det` |
| Temperature, humidity, cloud | `meps_mbr0_pp` | `meps_det` |

Only the 00 UTC runs are used: before 2020 the archive contains irradiance only in these files. Finished months are written to `data/nwp/monthly/`, so an interrupted run continues from the last month.

## Setup

In [ ]:
import os, sys

REPO = "helsinki-multihorizon-pv"
REPO_URL = f"https://github.com/rhmano/{REPO}.git"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO):
        os.system(f"git clone -q {REPO_URL}")
    ROOT = os.path.abspath(REPO)
    os.system(f"pip install -q netCDF4")
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

from multistep_core import *

## 1. Extraction
Missing files do not stop the extraction; they are logged in `data/nwp/failed_runs.csv` (`partial`: one file missing but variables taken from the other; `full`: nothing could be read). A run without irradiance is replaced by the previous day's run in the experiment.

In [ ]:
extract_meps_archive("2017-01-01", "2021-12-31")

## 2. Retrying failed runs
Temporary server errors are retried automatically; this cell retries the runs that could not be read at all.

In [ ]:
retry_failed_runs()

## 3. Merging and coverage

In [ ]:
NWP = build_nwp_table()
print("Irradiance coverage per year:")
print(NWP.groupby(NWP["run_time"].dt.year)["nwp_ghi"].apply(lambda x: x.notna().mean()).round(3))
if os.path.exists(CFG.nwp_failures_path):
    f = pd.read_csv(CFG.nwp_failures_path)
    print(f"\nFailed-run records: {len(f)} (full: {(f['type'] == 'full').sum()}, partial: {(f['type'] == 'partial').sum()})")

## 4. Time alignment
The files carry no `cell_methods`, so the shift is chosen from the correlation with measured GHI on 2017–2018 only (test years unseen). The result (−1 h) is the default `nwp_shift_hours` in `multistep_core.py`.

In [ ]:
BASE = prepare_base(CFG)
SHIFT = check_nwp_alignment(BASE, NWP)
BEST = check_nwp_alignment_by_source(BASE, NWP)